# 15 · Data cleaning

**What this notebook does:** it applies the cleaning decisions in **`docs/data_cleaning_decisions.md`** (the
specification) and saves a new, cleaned dataset. Every step below points to the section of that document it implements.

**What this notebook does NOT do:**
- no train/test split
- no model, no model preprocessing (imputation, encoding, scaling)
- no feature engineering
- no changes to the source files: `merged_listings.csv` and `localities.csv` are only **read**, and their MD5
  fingerprints are checked at the start and at the end

**Outputs (all new files):**

| File | Content |
|---|---|
| `data/processed/cleaned_listings.csv` | the cleaned dataset |
| `data/processed/cleaned_listings_column_roles.csv` | the role of every column (target, possible feature, metadata, …) |
| `data/processed/cleaning_removed_rows.csv` | every removed listing and the reason |
| `reports/cleaning_audit.csv` | row counts per step and the results of the validation checks |

**Order of the steps** (spec section 12). The order matters:

| Step | What | Why here |
|---|---|---|
| 0 | load, check fingerprints, check every row is a website *sale* listing | start from verified data |
| 1 | read a few helper fields from `raw_json` | needed by the group ID and the flags |
| 2 | build `property_group_id` v2 on the **unmodified** data | it uses area and floor, which steps 5b and 5c change |
| 3 | compute all flags (R1, R2, R3, missing price) on the **unmodified** data | R1 depends on the R2 flag |
| 4 | remove rows: missing price, impossible price (R1) | the only two row-removal rules |
| 5 | deterministic corrections: area unit conversion, R2, R3, one spelling variant | row-by-row, fixed rules |
| 6 | special missing representations → missing | placeholders are not real values |
| 7 | locality match status and recovery candidates | locality doc decisions |
| 8 | drop leaky / unnecessary columns, add `price_band`, record column roles | only columns the spec lists |
| 9 | final validation, save, reload, fingerprint check, audit | prove the result |

Every step only uses the listing's **own** values or rules fixed in advance. Nothing is learned from other listings'
prices, so doing this before the split does not leak information (spec section 12, "Why this order").

## 0. Setup and loading

**What:** import libraries, define file paths, and take an MD5 fingerprint of both source files.

**Why:** the MD5 fingerprint is a short code computed from every byte of a file. If a single byte changes, the
fingerprint changes. We compare it with the values recorded in the specification: if they differ, the counts in the
investigations may no longer hold and the notebook stops. At the end we check again that nothing was modified.

**Evidence:** spec header (source data table).

In [1]:
from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Source files: read only, never written
MERGED_CSV = PROJECT_ROOT / "data" / "processed" / "merged_listings.csv"
LOCALITIES_CSV = PROJECT_ROOT / "data" / "extracted" / "localities.csv"

# New files created by this notebook
CLEANED_CSV = PROJECT_ROOT / "data" / "processed" / "cleaned_listings.csv"
COLUMN_ROLES_CSV = PROJECT_ROOT / "data" / "processed" / "cleaned_listings_column_roles.csv"
REMOVED_ROWS_CSV = PROJECT_ROOT / "data" / "processed" / "cleaning_removed_rows.csv"
AUDIT_CSV = PROJECT_ROOT / "reports" / "cleaning_audit.csv"

# Fingerprints recorded in docs/data_cleaning_decisions.md
EXPECTED_MD5 = {
    "merged_listings.csv": "7d5c4224a1864259922d683567550231",
    "localities.csv": "8d9bb54dfc2fb49673e74b976ab73ace",
}


def file_md5(path):
    # A fingerprint of the file's bytes. If the file changes, the fingerprint changes.
    return hashlib.md5(path.read_bytes()).hexdigest()


md5_at_start = {"merged_listings.csv": file_md5(MERGED_CSV), "localities.csv": file_md5(LOCALITIES_CSV)}
print("MD5 at start:", md5_at_start)
assert md5_at_start == EXPECTED_MD5, "A source file differs from the one the specification was written for. Stop and investigate."
print("Both source files match the specification.")

MD5 at start: {'merged_listings.csv': '7d5c4224a1864259922d683567550231', 'localities.csv': '8d9bb54dfc2fb49673e74b976ab73ace'}
Both source files match the specification.


In [2]:
listings = pd.read_csv(MERGED_CSV, low_memory=False)
localities = pd.read_csv(LOCALITIES_CSV, low_memory=False)

original_row_count = len(listings)
original_column_count = listings.shape[1]
print("listings:  ", listings.shape)
print("localities:", localities.shape)

# property_id must identify each row exactly once; everything below relies on it
assert listings["property_id"].is_unique, "property_id is not unique"
print("property_id is unique:", listings["property_id"].is_unique)

listings:   (20694, 95)
localities: (2009, 38)
property_id is unique: True


In [3]:
# The audit collects one line per step. It is saved to reports/cleaning_audit.csv at the end.
audit = []


def add_to_audit(step, value, note=""):
    audit.append({"step": step, "value": value, "note": note})


add_to_audit("original rows", original_row_count, "data/processed/merged_listings.csv")
add_to_audit("original columns", original_column_count)

## 1. Read helper fields from `raw_json`

**What:** `raw_json` is the website's original record for each listing (169 fields). We read **only** these named fields:

| `raw_json` field | Used for |
|---|---|
| `transType` | the check that every listing is a sale listing (step 1b) |
| `ca` | the covered area exactly as typed (used by the flags, as in notebook 14) |
| `covAreaUnit`, `covAreaUnitDesc` | the **true** covered-area unit, as a code and as text |
| `caSqFt` | the website's own covered area in sq-ft (used by the R1/R2 flags and to cross-check our conversion) |
| `psmid` | project/society ID (kept as `project_id`, metadata) |
| `landAreaUnitD` | whether the website form had a separate land-area unit (kept as `land_area_unit`, a possible later "plot area" signal) |
| `oid` | advertiser ID (kept as `advertiser_id`, metadata only) |

**Why:** `raw_json` itself is **forbidden as a feature**, because it contains the price (`price`, `priceD`,
`sqFtPrice`). No price field is read here. The `raw_json` column is dropped in step 8.

**Evidence:** spec section 3h; notebook 08 (units); notebook 09 (`psmid`, `covAreaUnit`).

Not carried into the cleaned file: `caSqFt` (it would bring back the unreliable areas that R2 removes; our own
converted area replaces it), `carpAreaUnit` (carpet area is not used, spec section 6) and `postDateT` (EDA only,
spec section 8; it can be re-read from `merged_listings.csv` when needed).

In [4]:
def read_raw_field(raw_text, field_name):
    # Same helper as notebooks 09 and 14: the field as text, or None when it is absent or empty
    value = json.loads(raw_text).get(field_name)
    if value is None or str(value).strip() == "":
        return None
    return str(value)


RAW_FIELDS = ["transType", "ca", "covAreaUnit", "covAreaUnitDesc", "caSqFt", "psmid", "landAreaUnitD", "oid"]

raw = pd.DataFrame(index=listings.index)
for field in RAW_FIELDS:
    raw[field] = listings["raw_json"].apply(read_raw_field, field_name=field)

print("Rows where each raw_json field is present:")
print(raw.notna().sum().to_string())

Rows where each raw_json field is present:
transType          20694
ca                 19065
covAreaUnit        19076
covAreaUnitDesc    19076
caSqFt             19063
psmid               8801
landAreaUnitD       1688
oid                20694


### 1b. Every row must be a website sale listing

**What:** check that `transType == "Sale"` for every row.

**Why:** the specification keeps **all** transaction types (Resale, New Property, Other, Rent, missing), because the
website itself marks every listing as a sale, and notebook 05 found **0 genuine rentals**. The `Rent` label is a wrong
sub-label inside sale adverts. This check makes that assumption explicit: if a future re-scrape contained real rentals,
the notebook would stop here instead of silently mixing rents into the target.

**Evidence:** spec section 5 (nb05 option 1); notebook 05 section 7.

In [5]:
print("transType values:", raw["transType"].value_counts(dropna=False).to_dict())
assert (raw["transType"] == "Sale").all(), "Some rows are not website sale listings. Stop and re-check notebook 05."

print()
print("transaction_type labels (all kept; nothing is filtered on this column):")
print(listings["transaction_type"].value_counts(dropna=False).to_string())
add_to_audit("rows removed because of transaction_type", 0, "all rows are website sale listings (spec section 5)")

transType values: {'Sale': 20694}

transaction_type labels (all kept; nothing is filtered on this column):
transaction_type
Resale          17759
New Property     2797
NaN                63
Other              57
Rent               18


## 2. Build `property_group_id` v2 on the unmodified data

**What:** give every listing a group ID so that copies of the same property can be kept on the same side of the
later train/test split:

```
property_group_id = location | property_type | bedrooms | bathrooms | covered area in sq-ft | floor
```

Listings that cannot be compared (no location, no area, no bedrooms, or an unreliable unit) get `single <property_id>`,
a group of their own.

**Why now, before any correction:** the ID uses **area** and **floor**. Steps 5b (R2) and 5c (R3) set some of those
values to missing. If we built the ID afterwards, those listings would become `single` groups and could be separated
from their copies. Version 2 was verified on the raw values (8 checks in notebook 09), so we rebuild it on exactly the
same raw values.

**How:** the three functions below are **copied unchanged** from notebook 09 (they are also copied unchanged in
notebook 14). Price and advertiser are deliberately **not** part of the ID (a repost at a new price, or by another
agent, is still the same property).

**Evidence:** spec sections 9 and 12; `docs/duplicate_handling.md`; notebook 09.

In [6]:
# ---- copied unchanged from notebooks/09_property_group_id_v2.ipynb ----
# Website unit code -> square feet per unit. Codes not listed here are treated as unreliable.
SQFT_PER_UNIT_CODE = {
    "12800": 1.0,       # Sq-ft
    "12850": 1.0,       # Sq-ft (land-area version)
    "12803": 9.0,       # Sq-yrd
    "12851": 9.0,       # Sq-yrd (land-area version)
    "12801": 10.7639,   # Sq-m
}

SIGNATURE_PARTS = ["location", "property_type", "bedrooms", "bathrooms", "area_sqft", "floor"]


def build_location(listings):
    project_id = listings["raw_json"].apply(read_raw_field, field_name="psmid")
    locality = listings["locality"].str.lower().str.strip()
    location = pd.Series(np.where(project_id.notna(), "project " + project_id.astype(str),
                                  "locality " + locality.astype(str)), index=listings.index)
    location[project_id.isna() & locality.isna()] = None
    return location


def build_property_group_id_v2(listings):
    parts = pd.DataFrame(index=listings.index)
    parts["location"] = build_location(listings)

    # A missing value is written as the word "missing", so the ID text never contains NaN
    parts["property_type"] = listings["property_type"].fillna("missing").astype(str)
    parts["bedrooms"] = listings["bedrooms"].fillna("missing").astype(str)
    parts["bathrooms"] = listings["bathrooms"].fillna("missing").astype(str)
    parts["floor"] = listings["floor"].fillna("missing").astype(str)

    # Covered area: typed number (commas removed) x factor for the TRUE unit from raw_json
    typed_area = pd.to_numeric(listings["covered_area_sqft"].astype(str).str.replace(",", ""), errors="coerce")
    unit_code = listings["raw_json"].apply(read_raw_field, field_name="covAreaUnit")
    # Codes can come back as "12800.0"; keep just the digits before the decimal point
    unit_code = unit_code.str.split(".").str[0]
    sqft_factor = unit_code.map(SQFT_PER_UNIT_CODE)      # NaN when the unit is missing or unreliable
    parts["area_sqft"] = (typed_area * sqft_factor).round(1)

    text_parts = pd.DataFrame(index=listings.index)
    for part in SIGNATURE_PARTS:
        text_parts[part] = parts[part].fillna("missing").astype(str)
    group_id = text_parts[SIGNATURE_PARTS].agg(" | ".join, axis=1)

    # Not comparable: no location, no area, no bedrooms, or an unreliable/missing unit -> own group
    can_compare = parts["location"].notna() & parts["area_sqft"].notna() & listings["bedrooms"].notna()
    group_id[~can_compare] = "single " + listings.loc[~can_compare, "property_id"].astype(str)
    return group_id
# ---- end of copy ----

listings["property_group_id"] = build_property_group_id_v2(listings)

**Validation:** the result must match the numbers recorded in `docs/duplicate_handling.md` (section 5) exactly:
19,851 groups, 2,404 `single` groups, largest group 10 listings.

In [7]:
group_ids = listings["property_group_id"]
group_sizes = group_ids.value_counts()
is_single = group_ids.str.startswith("single ")

print("Listings:                     ", len(group_ids))
print("Missing or empty IDs:         ", (group_ids.isna() | (group_ids == "")).sum())
print("Unique property_group_id:     ", group_ids.nunique())
print("'single' groups:              ", is_single.sum(), "| all unique:", group_ids[is_single].is_unique)
print("Largest group:                ", group_sizes.max(), "listings")
print("Listings in multi-listing groups:", (group_ids.map(group_sizes) > 1).sum())
print()
print("Listings per group -> number of groups:")
print(group_sizes.value_counts().sort_index().to_string())

assert group_ids.notna().all() and (group_ids != "").all()
assert group_ids.nunique() == 19851, "Group count differs from docs/duplicate_handling.md. Stop and investigate."
assert is_single.sum() == 2404
assert group_ids[is_single].is_unique
assert group_sizes.max() == 10
print()
print("property_group_id v2 matches the documented result.")
add_to_audit("unique property_group_id (all 20,694 rows)", group_ids.nunique(), "matches docs/duplicate_handling.md")

Listings:                      20694
Missing or empty IDs:          0
Unique property_group_id:      19851
'single' groups:               2404 | all unique: True
Largest group:                 10 listings
Listings in multi-listing groups: 1568

Listings per group -> number of groups:
count
1     19126
2       645
3        63
4         8
5         4
6         1
7         3
10        1

property_group_id v2 matches the documented result.


## 3. Compute all flags on the unmodified data

**What:** mark (True/False) which rows each cleaning rule applies to. **Nothing is changed or removed yet.**

**Why all flags first:**
- R1 (impossible price) depends on R2: an extreme price per sq-ft only counts as a **price** error when the **area**
  is not already known to be unreliable.
- Computing every flag on the same untouched table makes each count traceable to notebook 14.

**How:** the definitions are copied from notebook 14 (sections 1–6 and 11). They use only the listing's own values and
fixed thresholds written down in advance, never statistics of the dataset.

A note on `price_per_sqft`: it is price ÷ covered sq-ft, so it contains the target. It is used **here only**, inside a
row-level check with fixed thresholds, and it is dropped in step 8. It never becomes a feature (spec section 4).

**Evidence:** spec sections 2, 6 and 11; `docs/outlier_handling.md` Part C; notebook 14 section 11.

In [8]:
# Plain inputs for the flags (same definitions as notebook 14)
price = listings["price"]
bedrooms_number = pd.to_numeric(listings["bedrooms"], errors="coerce")    # the text "> 10" becomes NaN, as in notebook 14
covered_typed = pd.to_numeric(raw["ca"], errors="coerce")                  # covered area exactly as typed
covered_unit = raw["covAreaUnitDesc"]                                      # the TRUE covered-area unit (not area_type)
covered_sqft_website = pd.to_numeric(raw["caSqFt"], errors="coerce")      # website's covered area in sq-ft
rate = listings["price_per_sqft"]                                          # CLEANING CHECK ONLY: derived from the target

is_house_like = listings["property_type"].isin(["Residential House", "Villa"])
reliable_unit = covered_unit.isin(["Sq-ft", "Sq-yrd", "Sq-m"])

flags = pd.DataFrame(index=listings.index)

### 3a. R2 flags: unreliable area

A row's covered area is unreliable if **any** of these is true (spec section 6):

| Component | Condition |
|---|---|
| land unit / no unit | a typed covered area exists, but its unit is not Sq-ft, Sq-yrd or Sq-m |
| small-house cluster | house/villa, true unit Sq-ft, typed value < 400 (very likely square yards typed as sq-ft; cannot be proven per row) |
| tiny area | reliable unit and website sq-ft < 200 |
| huge area | reliable unit and website sq-ft > 20,000 |
| too much area per bedroom | reliable unit and sq-ft per bedroom > 5,000 |

In [9]:
flags["land_unit_area"] = covered_typed.notna() & ~reliable_unit
flags["small_house_sqft_cluster"] = is_house_like & (covered_unit == "Sq-ft") & (covered_typed < 400)
flags["area_below_200"] = reliable_unit & (covered_sqft_website < 200)
flags["area_above_20000"] = reliable_unit & (covered_sqft_website > 20000)
sqft_per_bedroom = covered_sqft_website / bedrooms_number
flags["over_5000_sqft_per_bedroom"] = reliable_unit & (sqft_per_bedroom > 5000)

r2_area_unreliable = (flags["land_unit_area"] | flags["small_house_sqft_cluster"] | flags["area_below_200"]
                      | flags["area_above_20000"] | flags["over_5000_sqft_per_bedroom"])

for flag_name in ["land_unit_area", "small_house_sqft_cluster", "area_below_200", "area_above_20000",
                  "over_5000_sqft_per_bedroom"]:
    print(f"{flag_name:30s} {flags[flag_name].sum():5d}")
print(f"{'R2 union (area unreliable)':30s} {r2_area_unreliable.sum():5d}")

land_unit_area                    10
small_house_sqft_cluster         210
area_below_200                   149
area_above_20000                  22
over_5000_sqft_per_bedroom        35
R2 union (area unreliable)       274


### 3b. R1 flags: impossible price

A row is removed by R1 if **any** of these is true (spec section 2):

| Component | Condition |
|---|---|
| R1a | price ≥ ₹100 crore |
| R1b | Telecalling ₹3,300 batch: in the small-house cluster **and** ₹3,200 ≤ price per sq-ft ≤ ₹3,400 (price = area × a fixed rate, so both are wrong) |
| R1c | price per sq-ft > ₹1 lakh, or (reliable unit and price per sq-ft < ₹1,000), **and** the area is **not** unreliable (R2) |

In [10]:
flags["price_100cr_plus"] = price >= 100e7                                  # ₹100 crore = 1,000,000,000
flags["telecalling_3300_batch"] = flags["small_house_sqft_cluster"] & rate.between(3200, 3400)
flags["rate_above_1_lakh"] = rate > 1e5
flags["rate_below_1000"] = reliable_unit & (rate < 1000)

extreme_rate_price_error = (flags["rate_above_1_lakh"] | flags["rate_below_1000"]) & ~r2_area_unreliable

r1_impossible_price = flags["price_100cr_plus"] | flags["telecalling_3300_batch"] | extreme_rate_price_error

print("R1a price >= ₹100 Cr:                          ", flags["price_100cr_plus"].sum())
print("R1b Telecalling ₹3,300 batch:                  ", flags["telecalling_3300_batch"].sum())
print("R1c extreme rate, no area problem, < ₹100 Cr:  ",
      (extreme_rate_price_error & ~flags["price_100cr_plus"]).sum())
print("R1 total (impossible price):                   ", r1_impossible_price.sum())

R1a price >= ₹100 Cr:                           14
R1b Telecalling ₹3,300 batch:                   29
R1c extreme rate, no area problem, < ₹100 Cr:   71
R1 total (impossible price):                    114


### 3c. R3 flag: contradictory floor, and the missing-price flag

- **R3:** the floor number is higher than the building's total floors. `Ground` = 0, `Upper Basement` = −1,
  `Lower Basement` = −2 when comparing. Only this internal contradiction counts; the 7 building heights contradicted by
  other listings of the same project stay unchanged (spec section 11).
- **Missing price (D1):** no target, so the row cannot be used for training or evaluation (spec section 2).

In [11]:
FLOOR_WORDS = {"Ground": "0", "Upper Basement": "-1", "Lower Basement": "-2"}
floor_number = pd.to_numeric(listings["floor"].replace(FLOOR_WORDS), errors="coerce")
r3_floor_contradiction = floor_number > listings["total_floors"]

d1_missing_price = price.isna()

print("R3 floor > total floors:", r3_floor_contradiction.sum())
print("D1 missing price:       ", d1_missing_price.sum())
print("  of which price_label: ", listings.loc[d1_missing_price, "price_label"].value_counts(dropna=False).to_dict())

R3 floor > total floors: 6
D1 missing price:        280
  of which price_label:  {'Call for Price': 263, nan: 17}


**Validation:** because the source file's MD5 matches, every count must equal the number recorded in the
specification / notebook 14. If any differs, the notebook stops, because a rule would then not be doing what was documented.

In [12]:
DOCUMENTED_COUNTS = {
    "R2 land unit / no unit": (flags["land_unit_area"].sum(), 10),
    "R2 small-house cluster": (flags["small_house_sqft_cluster"].sum(), 210),
    "R2 area < 200 sq-ft": (flags["area_below_200"].sum(), 149),
    "R2 area > 20,000 sq-ft": (flags["area_above_20000"].sum(), 22),
    "R2 > 5,000 sq-ft per bedroom": (flags["over_5000_sqft_per_bedroom"].sum(), 35),
    "R2 union": (r2_area_unreliable.sum(), 274),
    "R1a price >= ₹100 Cr": (flags["price_100cr_plus"].sum(), 14),
    "R1b Telecalling batch": (flags["telecalling_3300_batch"].sum(), 29),
    "R1c extreme rate, no area problem": ((extreme_rate_price_error & ~flags["price_100cr_plus"]).sum(), 71),
    "R1 total": (r1_impossible_price.sum(), 114),
    "R3 floor > total floors": (r3_floor_contradiction.sum(), 6),
    "D1 missing price": (d1_missing_price.sum(), 280),
    "R1-R3 rows touched": ((r1_impossible_price | r2_area_unreliable | r3_floor_contradiction).sum(), 365),
}

count_check = pd.DataFrame(DOCUMENTED_COUNTS, index=["this notebook", "documented"]).T
count_check["same?"] = count_check["this notebook"] == count_check["documented"]
print(count_check.to_string())
assert count_check["same?"].all(), "A rule count differs from the specification. Stop and report."
print()
print("All rule counts match the specification.")

# R1 needs a price, so the two removal rules cannot overlap
print("Rows flagged by both D1 and R1:", (d1_missing_price & r1_impossible_price).sum())
assert (d1_missing_price & r1_impossible_price).sum() == 0

                                   this notebook  documented  same?
R2 land unit / no unit                        10          10   True
R2 small-house cluster                       210         210   True
R2 area < 200 sq-ft                          149         149   True
R2 area > 20,000 sq-ft                        22          22   True
R2 > 5,000 sq-ft per bedroom                  35          35   True
R2 union                                     274         274   True
R1a price >= ₹100 Cr                          14          14   True
R1b Telecalling batch                         29          29   True
R1c extreme rate, no area problem             71          71   True
R1 total                                     114         114   True
R3 floor > total floors                        6           6   True
D1 missing price                             280         280   True
R1-R3 rows touched                           365         365   True

All rule counts match the specification.
Rows f

## 4. Remove rows: missing price (D1) and impossible price (R1)

**What:** remove the 280 rows without a price and the 114 rows with an impossible price.

**Why:** these are the **only** two reasons a row is removed, and both concern the **target**:
- a row with no price cannot be used for training or evaluation;
- an impossible price (e.g. a 2-bedroom flat at ₹1,216 Cr) is a typing error that cannot be corrected. Keeping it would
  reward the model for predicting typos. Setting it to missing would leave a row without a target, so the row is removed.

Nothing is removed for being unusual (luxury homes, 10+ bedroom houses, tall towers stay), for its transaction type,
for being a suspected duplicate, for an unmatched locality, or for missing detail fields.

R1 is applied before the split, so it also affects the future test set. This must be stated when reporting results
("listings with impossible asking prices, ≈ 0.6%, were excluded").

The removed listings are saved to `data/processed/cleaning_removed_rows.csv` so every removal can be traced.

**Evidence:** spec section 2; `docs/missingness_handling.md` B22; `docs/outlier_handling.md` R1.

In [13]:
# First attach the columns that must travel with the rows: the group ID was added in step 2,
# the R2/R3 flags are kept so every changed value can be traced later (spec section 3g)
listings["r2_area_unreliable"] = r2_area_unreliable
listings["r3_floor_contradiction"] = r3_floor_contradiction

# Build the list of removed rows, with the reason for each
removal_reason = pd.Series("", index=listings.index)
removal_reason[d1_missing_price] = "D1 missing price"
removal_reason[flags["price_100cr_plus"]] = "R1a price >= 100 Cr"
removal_reason[flags["telecalling_3300_batch"]] = "R1b Telecalling 3,300 per sq-ft batch"
removal_reason[extreme_rate_price_error & ~flags["price_100cr_plus"]] = "R1c extreme price per sq-ft, area plausible"

is_removed = d1_missing_price | r1_impossible_price

REMOVED_COLUMNS_TO_SAVE = ["property_id", "property_group_id", "price", "price_label", "property_type", "bedrooms",
                           "covered_area_sqft", "price_per_sqft", "transaction_type", "pmt_source", "locality"]
removed_rows = listings.loc[is_removed, REMOVED_COLUMNS_TO_SAVE].copy()
removed_rows.insert(1, "removal_reason", removal_reason[is_removed])
removed_rows["covered_area_unit"] = covered_unit[is_removed]
removed_rows.to_csv(REMOVED_ROWS_CSV, index=False)

print("Removed rows by reason:")
print(removed_rows["removal_reason"].value_counts().to_string())
print("Saved to:", REMOVED_ROWS_CSV.relative_to(PROJECT_ROOT))

Removed rows by reason:
removal_reason
D1 missing price                               280
R1c extreme price per sq-ft, area plausible     71
R1b Telecalling 3,300 per sq-ft batch           29
R1a price >= 100 Cr                             14
Saved to: data\processed\cleaning_removed_rows.csv


In [14]:
# Keep everything else. .copy() makes an independent table, so later changes never touch `listings`.
cleaned = listings.loc[~is_removed].copy()

print("Rows before:             ", len(listings))
print("Removed, missing price:  ", d1_missing_price.sum())
print("Removed, impossible price:", r1_impossible_price.sum())
print("Rows after:              ", len(cleaned))
assert len(cleaned) == original_row_count - d1_missing_price.sum() - r1_impossible_price.sum()

print()
print("transaction_type after removal (nothing filtered on it):")
print(cleaned["transaction_type"].value_counts(dropna=False).to_string())

add_to_audit("rows removed: missing price (D1)", int(d1_missing_price.sum()), "no target")
add_to_audit("rows removed: impossible price (R1)", int(r1_impossible_price.sum()),
             "R1a 14 / R1b 29 / R1c 71 (docs/outlier_handling.md R1)")
add_to_audit("rows after removal", len(cleaned))

Rows before:              20694
Removed, missing price:   280
Removed, impossible price: 114
Rows after:               20300

transaction_type after removal (nothing filtered on it):
transaction_type
Resale          17417
New Property     2779
Other              55
NaN                34
Rent               15


## 5. Deterministic corrections

Each correction below works row by row with a fixed rule. None of them uses other listings or learns anything from
the data.

### 5a. Covered area: remove commas and convert to square feet with the true unit

**What:**
1. `covered_area_sqft` is text and 108 values contain a thousands comma (`"1,215"`). Remove the commas and store the
   value as a number.
2. Create `covered_area_sqft_converted` = typed value × the factor for the **true** unit `raw_json.covAreaUnit`:
   Sq-ft × 1, Sq-yrd × 9, Sq-m × 10.7639. For any other unit (Ground, Cent, Kanal, Biswa, Hectare) or no unit, the
   converted value is **missing**.

**Why:**
- Despite its name, `covered_area_sqft` holds the value **as typed, in the advertiser's unit** (notebook 08). So "150"
  can mean 150 sq-ft or 150 sq-yd = 1,350 sq-ft. The column name is kept unchanged, and it keeps its original meaning
  (typed value, typed unit); `covered_area_sqft_converted` is the comparable square-foot value.
- The unit comes from `raw_json.covAreaUnit`, **never** from `area_type`: `area_type` holds the carpet-area unit whenever
  a carpet area exists and is wrong for 475 rows.
- The three factors are exactly the website's own (notebook 08 section 7). Land units convert to impossible sizes, so no
  factor is trusted for them.

This is the same conversion used inside `property_group_id` v2 (the same `SQFT_PER_UNIT_CODE` table). It only
standardises the unit; it does **not** decide how area will be used as a feature (spec section 6: plot vs built-up
area for houses is a feature-engineering question).

**Evidence:** spec section 6; notebook 08 sections 1, 2 and 7; notebook 09.

In [15]:
print("covered_area_sqft values containing a comma:", cleaned["covered_area_sqft"].str.contains(",", na=False).sum())

# 1. Typed value as a number (commas removed). Still in the advertiser's unit.
typed_area = pd.to_numeric(cleaned["covered_area_sqft"].str.replace(",", ""), errors="coerce")

# 2. Factor for the TRUE unit code. Codes can be stored as "12800.0", so keep the digits before the decimal point.
unit_code = raw.loc[cleaned.index, "covAreaUnit"].str.split(".").str[0]
sqft_factor = unit_code.map(SQFT_PER_UNIT_CODE)          # NaN for land units and missing units

cleaned["covered_area_sqft"] = typed_area
cleaned["covered_area_unit"] = covered_unit[cleaned.index]
cleaned["covered_area_sqft_converted"] = typed_area * sqft_factor

print("Typed covered area present:       ", typed_area.notna().sum())
print("Converted to sq-ft:               ", cleaned["covered_area_sqft_converted"].notna().sum())
print("Typed value but unit not reliable:", (typed_area.notna() & sqft_factor.isna()).sum(), "(these are all in R2)")
print()
print("Converted rows by true unit:")
print(cleaned.loc[cleaned["covered_area_sqft_converted"].notna(), "covered_area_unit"].value_counts().to_string())

covered_area_sqft values containing a comma: 106
Typed covered area present:        18792
Converted to sq-ft:                18782
Typed value but unit not reliable: 10 (these are all in R2)

Converted rows by true unit:
covered_area_unit
Sq-ft     16864
Sq-yrd     1909
Sq-m          9


**Validation:** the unit code (used for the conversion) and the unit text (used by the flags) must agree about which
units are reliable, and our square-foot value must equal the website's own `caSqFt`.

In [16]:
reliable_by_text = cleaned["covered_area_unit"].isin(["Sq-ft", "Sq-yrd", "Sq-m"])
reliable_by_code = sqft_factor.notna()
print("Rows where unit code and unit text disagree about reliability:", (reliable_by_text != reliable_by_code).sum())
assert (reliable_by_text == reliable_by_code).all()

website_sqft = covered_sqft_website[cleaned.index]
both = cleaned["covered_area_sqft_converted"].notna() & website_sqft.notna()
relative_difference = (cleaned["covered_area_sqft_converted"][both] / website_sqft[both] - 1).abs()
print("Rows compared with the website's caSqFt:", both.sum())
print("Largest relative difference:            ", f"{relative_difference.max():.5%}",
      "(Sq-m: the website uses 10.761-10.767 instead of 10.7639)")
assert relative_difference.max() < 0.001, "Our conversion disagrees with the website's sq-ft value"

Rows where unit code and unit text disagree about reliability: 0
Rows compared with the website's caSqFt: 18782
Largest relative difference:             0.02760% (Sq-m: the website uses 10.761-10.767 instead of 10.7639)


### 5b. R2: unreliable area → missing (row kept)

**What:** for the rows flagged by R2 in step 3a, set both `covered_area_sqft` (typed value) and
`covered_area_sqft_converted` to missing. The row itself stays.

**Why:** the price and other fields of these listings are plausible; only the area is not. Converting the small-house
cluster "× 9" would be a guess that cannot be proven for any single row, so **no unit is guessed or converted**. The
missing area will be handled later, inside the modelling pipeline, fitted on training data only. Covered area is
**not** filled from carpet area (a different measure). The flag column `r2_area_unreliable` stays so the change can be traced.

**Evidence:** spec sections 6 and 11; `docs/outlier_handling.md` R2; `docs/missingness_handling.md` B10.

In [17]:
r2_rows = cleaned["r2_area_unreliable"]
had_area_before = cleaned["covered_area_sqft"].notna()

cleaned.loc[r2_rows, "covered_area_sqft"] = np.nan
cleaned.loc[r2_rows, "covered_area_sqft_converted"] = np.nan

print("R2 rows in the cleaned data:           ", r2_rows.sum(), "(of 274 before row removal)")
print("  of which had a typed area before:    ", (r2_rows & had_area_before).sum())
print("Rows without a converted area, total:  ", cleaned["covered_area_sqft_converted"].isna().sum())
print("  already without a covered area:      ", (~had_area_before).sum())
print("  set to missing by R2:                ", (r2_rows & had_area_before).sum())

# After R2, every remaining typed area has a reliable unit, so typed and converted are missing on the same rows
assert (cleaned["covered_area_sqft"].notna() == cleaned["covered_area_sqft_converted"].notna()).all()
converted = cleaned["covered_area_sqft_converted"].dropna()
print("Converted area range after R2:", round(converted.min(), 1), "to", round(converted.max(), 1), "sq-ft")

add_to_audit("rows with area set to missing (R2)", int((r2_rows & had_area_before).sum()),
             "rows kept; no unit guessed (docs/outlier_handling.md R2)")
add_to_audit("rows without usable covered area", int(cleaned["covered_area_sqft_converted"].isna().sum()),
             "no covered area at all, or R2")

R2 rows in the cleaned data:            230 (of 274 before row removal)
  of which had a typed area before:     230
Rows without a converted area, total:   1738
  already without a covered area:       1508
  set to missing by R2:                 230
Converted area range after R2: 200.0 to 20000.0 sq-ft


### 5c. R3: contradictory floor → `floor` and `total_floors` missing (row kept)

**What:** for the 6 rows where the floor is above the building's total floors, set both fields to missing.

**Why:** the combination is impossible, and we cannot tell which of the two numbers is wrong. Other unusually high
values (including the 7 heights contradicted by other listings in the same project) are **not** changed, because no
observation-level rule shows they are invalid.

Note: `property_group_id` was built in step 2 from the original floor values, so these rows keep their group.

**Evidence:** spec section 11; `docs/outlier_handling.md` R3.

In [18]:
r3_rows = cleaned["r3_floor_contradiction"]
print("R3 rows before the change:")
print(cleaned.loc[r3_rows, ["property_id", "property_type", "floor", "total_floors"]].to_string())

cleaned.loc[r3_rows, "floor"] = np.nan
cleaned.loc[r3_rows, "total_floors"] = np.nan

print()
print("R3 rows with floor and total_floors now missing:",
      (cleaned.loc[r3_rows, "floor"].isna() & cleaned.loc[r3_rows, "total_floors"].isna()).sum())

# Re-check: no remaining floor is above its total floors
floor_number_after = pd.to_numeric(cleaned["floor"].replace(FLOOR_WORDS), errors="coerce")
print("Remaining rows with floor > total floors:", (floor_number_after > cleaned["total_floors"]).sum())
assert (floor_number_after > cleaned["total_floors"]).sum() == 0

add_to_audit("rows with floor fields set to missing (R3)", int(r3_rows.sum()), "floor > total floors")

R3 rows before the change:
       property_id      property_type floor  total_floors
10690  listing_001  Residential House     3           2.0
16032  listing_002  Residential House     3           2.0
20383  listing_003  Residential House     5           4.0

R3 rows with floor and total_floors now missing: 3
Remaining rows with floor > total floors: 0


### 5d. Spelling variant of the same category

**What:** replace `"Ready To Move"` with `"Ready to Move"` in `possession`.

**Why:** they are the same category written with different capital letters (notebook 03). Left as is, they would
later become two separate categories. This is the only spelling variant named in the specification, so it is the only
one changed. The cell below then **lists** any other case-only variants in the kept category columns **without
changing them**, so they can be decided explicitly rather than silently.

**Evidence:** spec section 12 (step 5); notebook 03 findings ("Text categories").

In [19]:
respelled = cleaned["possession"] == "Ready To Move"
print("Before: 'Ready to Move':", (cleaned["possession"] == "Ready to Move").sum(), "| 'Ready To Move':", respelled.sum())

cleaned["possession"] = cleaned["possession"].replace({"Ready To Move": "Ready to Move"})

print("After:  'Ready to Move':", (cleaned["possession"] == "Ready to Move").sum(),
      "| 'Ready To Move':", (cleaned["possession"] == "Ready To Move").sum())
add_to_audit("possession values respelled ('Ready To Move' -> 'Ready to Move')", int(respelled.sum()), "notebook 03")

Before: 'Ready to Move': 18033 | 'Ready To Move': 20
After:  'Ready to Move': 18053 | 'Ready To Move': 0


In [20]:
# Look for other values that differ only in capital letters or surrounding spaces. Report only; nothing is changed.
CATEGORY_COLUMNS = ["property_type", "transaction_type", "furnishing", "possession", "age_of_property", "facing",
                    "posted_by", "prime_location", "balconies", "bedrooms", "bathrooms"]

found_any = False
for column in CATEGORY_COLUMNS:
    spellings_per_value = {}
    for value in cleaned[column].dropna().unique():
        key = str(value).lower().strip()
        if key not in spellings_per_value:
            spellings_per_value[key] = []
        spellings_per_value[key].append(value)
    for key, spellings in spellings_per_value.items():
        if len(spellings) > 1:
            found_any = True
            print(f"{column}: {spellings}")

if not found_any:
    print("No other case-only spelling variants in these columns.")

No other case-only spelling variants in these columns.


## 6. Special missing representations → missing

**What:** some values are stored as ordinary data but **mean "missing"**. Convert them to real missing values (`NaN`):

| Column(s) | Stored value | Why it is not a real value |
|---|---|---|
| `latitude`, `longitude` | both `0` | (0, 0) is in the Atlantic Ocean: the website's placeholder for "no map pin" |
| 16 `loc_rating_*` columns | `0` | 0 means "locality not rated". Read as a number it would say half the localities are the worst possible |
| `nonluxAmenMap` | `{}` | an empty map, not a list of amenities |
| `sub_locality` | `Na` | text for "no value" |

**Why now:** this is a fixed, row-by-row rule, not an imputation. Nothing is filled in; the values only become honestly
missing. How missing values are later filled or encoded is decided in the modelling pipeline on training data only.

**Not changed:** coordinates are **never** swapped, repaired or moved (e.g. the 4 swapped pairs or pins outside
Hyderabad stay as they are). The final definition of "unusable coordinates" (R4) is still open and is only needed if
geographic features are used. `loc_review_count` = 0 is a genuine count and is not changed.

**Evidence:** spec sections 7, 10 (step 1) and 11 (R4); `docs/missingness_handling.md` A2, B1–B3;
`docs/locality_handling.md` B9, B14.

In [21]:
# (0, 0) coordinates -> missing. Only when BOTH are 0 (that is the placeholder).
zero_coordinates = (cleaned["latitude"] == 0) & (cleaned["longitude"] == 0)
only_one_zero = ((cleaned["latitude"] == 0) | (cleaned["longitude"] == 0)) & ~zero_coordinates
print("(0, 0) placeholder coordinates:", zero_coordinates.sum())
print("Rows where only one of the two is 0:", only_one_zero.sum())

cleaned.loc[zero_coordinates, "latitude"] = np.nan
cleaned.loc[zero_coordinates, "longitude"] = np.nan
print("latitude/longitude equal to 0 after the change:",
      ((cleaned["latitude"] == 0) | (cleaned["longitude"] == 0)).sum())
add_to_audit("(0, 0) coordinates set to missing", int(zero_coordinates.sum()), "placeholder; no coordinate repaired")

(0, 0) placeholder coordinates: 7459
Rows where only one of the two is 0: 0
latitude/longitude equal to 0 after the change: 0


In [22]:
# Locality ratings: 0 means "not rated". loc_rating_users_count is a count, not a rating, so it is not in this list.
RATING_COLUMNS = ["loc_rating_safety", "loc_rating_cleanliness", "loc_rating_traffic", "loc_rating_connectivity",
                  "loc_rating_environment", "loc_rating_school", "loc_rating_hospital", "loc_rating_overall",
                  "loc_rating_road", "loc_rating_neighbour", "loc_rating_transport", "loc_rating_parking",
                  "loc_rating_restaurant", "loc_rating_market", "loc_rating_commuting", "loc_rating_placeOfInterest"]

rating_changes = {}
for column in RATING_COLUMNS:
    is_zero = cleaned[column] == 0
    rating_changes[column] = is_zero.sum()
    cleaned.loc[is_zero, column] = np.nan

print("Rating values 0 -> missing, per column:")
print(pd.Series(rating_changes).to_string())
print("Rating values still equal to 0:", (cleaned[RATING_COLUMNS] == 0).sum().sum())
add_to_audit("locality rating values 0 -> missing (16 columns, total)", int(sum(rating_changes.values())),
             f"loc_rating_overall: {rating_changes['loc_rating_overall']} rows")

Rating values 0 -> missing, per column:
loc_rating_safety             3110
loc_rating_cleanliness        3128
loc_rating_traffic            3153
loc_rating_connectivity       3141
loc_rating_environment        3110
loc_rating_school             3132
loc_rating_hospital           3413
loc_rating_overall            3110
loc_rating_road               3114
loc_rating_neighbour          3137
loc_rating_transport          3405
loc_rating_parking            3122
loc_rating_restaurant         3143
loc_rating_market             3135
loc_rating_commuting          3118
loc_rating_placeOfInterest    3128
Rating values still equal to 0: 0


In [23]:
empty_map = cleaned["nonluxAmenMap"] == "{}"
cleaned.loc[empty_map, "nonluxAmenMap"] = np.nan
print("nonluxAmenMap '{}' -> missing:", empty_map.sum())

na_text = cleaned["sub_locality"] == "Na"
cleaned.loc[na_text, "sub_locality"] = np.nan
print("sub_locality 'Na' -> missing: ", na_text.sum())

add_to_audit("nonluxAmenMap '{}' -> missing", int(empty_map.sum()))
add_to_audit("sub_locality 'Na' -> missing", int(na_text.sum()))

nonluxAmenMap '{}' -> missing: 5433
sub_locality 'Na' -> missing:  1


## 7. Locality handling

**What the specification decides** (spec section 7; `docs/locality_handling.md` Part B):

| Decision | What this notebook does |
|---|---|
| `locality_id` is the locality identity; the `locality` text is not a category | nothing to change; `locality_id` is kept as is |
| keep all unmatched listings; never drop or guess them | no row is removed or filled for its locality |
| the 41 listings whose ID is not in the locality table keep their ID | nothing to change |
| record the cause of a missing locality | new column `locality_match_status` (4 values, exactly as in notebook 10) |
| unknown-locality category, rare-ID grouping | **not here**: these are encodings fitted on training data later |
| never fuzzy-match or substring-match names | not done |

**The free-text recovery (spec open question 3).** Notebook 10 found that ≈ 110 free-text listings can be matched
safely: after the "level 2" normalisation (lower case, spaces collapsed, a 6-digit PIN removed, a trailing
"Hyderabad/Secunderabad/Telangana" removed) the text equals the name of **exactly one** `locality_id`. Whether to
**apply** these matches is still an open question. Decision for this notebook (confirmed 2026-09-27): **record, don't
apply**. The matched ID is stored in a separate audit column, `recovered_locality_id_candidate`, and `locality_id` and
`locality_match_status` stay unchanged. The candidates can be applied later with one line if the open question is
resolved.

The normalisation functions and the list of known names are copied from notebook 10 (section 2.3). The known names are
built from the **full** source data, exactly as in notebook 10. They are locality names only, no prices, so this lookup
cannot leak the target.

**Evidence:** `docs/locality_handling.md` A2, A4, B1–B6, C1; notebook 10 sections 1.2 and 2.3.

In [24]:
# Match status, exactly as in notebook 10 (section 1.2)
has_id = cleaned["locality_id"].notna()
matched = cleaned["locality_matched"]

locality_match_status = pd.Series("matched", index=cleaned.index)
locality_match_status[~has_id & cleaned["locality"].notna()] = "free text, no ID"
locality_match_status[~has_id & cleaned["locality"].isna()] = "no locality at all"
locality_match_status[has_id & ~matched] = "ID not in locality table"
cleaned["locality_match_status"] = locality_match_status

STATUS_ORDER = ["matched", "ID not in locality table", "free text, no ID", "no locality at all"]
print("Locality match status (cleaned rows):")
print(cleaned["locality_match_status"].value_counts().reindex(STATUS_ORDER).to_string())
print("Not matched in total:", (cleaned["locality_match_status"] != "matched").sum(),
      "- all kept, none guessed")

Locality match status (cleaned rows):
locality_match_status
matched                     18790
ID not in locality table       41
free text, no ID              585
no locality at all            884
Not matched in total: 1510 - all kept, none guessed


In [25]:
# ---- copied from notebooks/10_locality_investigation.ipynb (section 2.3) ----
def normalise_level_1(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def normalise_level_2(text):
    text = normalise_level_1(text)
    text = re.sub(r"\b\d{6}\b", "", text)  # PIN code
    text = re.sub(r"[,\s]*(hyderabad|secunderabad|telangana)[,\s]*$", "", text)
    return text.strip(" ,.-")
# ---- end of copy ----


# Known names, built as in notebook 10: display names of source listings that have an ID,
# plus the names in the Hyderabad part of the locality table
listings_with_id = listings[listings["locality_id"].notna()]
localities_hyd = localities[localities["city_name"] == "Hyderabad"]
known_names = pd.concat([
    listings_with_id[["locality", "locality_id"]].rename(columns={"locality": "name"}),
    localities_hyd[["locality_name", "locality_id"]].rename(columns={"locality_name": "name"}),
]).drop_duplicates()
known_names["normalised"] = known_names["name"].apply(normalise_level_2)

# A normalised name is safe only if it belongs to exactly ONE locality_id
ids_per_name = known_names.groupby("normalised")["locality_id"].nunique()
safe_names = ids_per_name[ids_per_name == 1].index
id_for_safe_name = (known_names[known_names["normalised"].isin(safe_names)]
                    .drop_duplicates("normalised")
                    .set_index("normalised")["locality_id"])
print("Known names:", len(known_names), "| normalised names pointing to exactly one ID:", len(safe_names))

Known names: 2743 | normalised names pointing to exactly one ID: 2599


In [26]:
def recovery_candidates(table, match_status):
    # Returns the single locality_id for safely matched free-text rows, and NaN for every other row
    is_free_text = match_status == "free text, no ID"
    normalised_text = table["locality"].apply(normalise_level_2)
    is_safe = is_free_text & normalised_text.isin(safe_names) & (normalised_text != "")
    candidate = pd.Series(np.nan, index=table.index)
    candidate[is_safe] = normalised_text[is_safe].map(id_for_safe_name)
    return candidate


# Check against notebook 10 on the full source data first (documented: 110 safe matches among 596 free-text rows)
full_has_id = listings["locality_id"].notna()
full_status = pd.Series("matched", index=listings.index)
full_status[~full_has_id & listings["locality"].notna()] = "free text, no ID"
full_status[~full_has_id & listings["locality"].isna()] = "no locality at all"
full_status[full_has_id & ~listings["locality_matched"]] = "ID not in locality table"
full_candidates = recovery_candidates(listings, full_status)
print("Safe recovery candidates in the full source data:", full_candidates.notna().sum(), "(documented: 110)")
assert full_candidates.notna().sum() == 110

# Now for the cleaned rows: stored in a separate column, NOT applied to locality_id
cleaned["recovered_locality_id_candidate"] = recovery_candidates(cleaned, cleaned["locality_match_status"])
print("Safe recovery candidates in the cleaned data:   ", cleaned["recovered_locality_id_candidate"].notna().sum())
print("locality_id filled by the recovery:             ", 0, "(recorded, not applied)")
print()
print("Examples:")
print(cleaned.loc[cleaned["recovered_locality_id_candidate"].notna(),
                  ["property_id", "locality", "recovered_locality_id_candidate"]].head(8).to_string())

add_to_audit("locality: free-text recovery candidates recorded (not applied)",
             int(cleaned["recovered_locality_id_candidate"].notna().sum()),
             "level-2 safe rule, exactly one ID; locality_id unchanged (spec open question 3)")
add_to_audit("locality: listings without a matched locality (kept)",
             int((cleaned["locality_match_status"] != "matched").sum()), "explicit unknown category later")

Safe recovery candidates in the full source data: 110 (documented: 110)
Safe recovery candidates in the cleaned data:    107
locality_id filled by the recovery:              0 (recorded, not applied)

Examples:
      property_id                       locality  recovered_locality_id_candidate
137   listing_004           Korremula, Hyderabad                          93716.0
401   listing_005            Neredmet, Hyderabad                          85804.0
1047  listing_006                       Bacharam                         113088.0
1051  listing_007                       Bacharam                         113088.0
2748  listing_008  Rampally Main Road, Hyderabad                         166519.0
2880  listing_009                     Bachupally                          84654.0
3132  listing_010              Nagole, Hyderabad                          89109.0
3155  listing_011                Kousalya Colony                         124907.0


## 8. Drop leaky and unnecessary columns; record the role of every other column

**What:** drop **only** the columns that the specification classifies as *forbidden (leakage)* or *definitely
remove*, plus `detail_url`, which the specification marks as "not needed in the cleaned file".

| Group | Columns | Why (spec section 3) |
|---|---|---|
| Forbidden: target-derived | `price_per_sqft`, `price_label`, `description`, `raw_json` | contain the price (3b, section 4) |
| Forbidden: website price statistics | `loc_avg_price_sqft`, `loc_avg_price_lmt`, `loc_price_quarter_prev`, `loc_price_quarter_current`, `loc_growth_quarterly_pct`, `loc_locality_rank` | asking-price averages from the same website, very likely including test listings; rank undocumented (3b) |
| Forbidden: collection process | `scraped_at` | scrape order ≈ price order (Spearman 0.985) (3b) |
| Definitely remove | `city`, `is_Luxury`, `detail_scraped`, `has_rera`, `legal_status`, `infra_status`, `loc_city_name`, `loc_city_id`, `bhk`, `property_title`, `coordinates`, `loc_lat`, `loc_long`, `luxAmenMap`, `lux_amen_desc`, `loc_locality_name`, `area_type`, `loc_scraped_at` | constant, broken, empty, exact copies, or (`area_type`) a misleading unit (3c) |
| Metadata not needed | `detail_url` | a second identifier (3f) |

**Kept, although not planned as features:** the *probably remove* columns (e.g. `project_name`, `carpet_area_sqft`,
`power_status`) and the *metadata only* columns (e.g. `property_id`, `listing_date`, `pmt_source`). The specification
does not remove them outright, so they stay in the file, and the **column-role file** says clearly that they are not
features.

**Also added:** `price_band`, the notebook 12 search range of each listing's **own** price, using the exact bins of
notebook 13 (a price exactly on a bound goes to the lower range). It is derived from the target, so it is for
**reporting errors per price range only** and must never be a feature (spec sections 3g, 4 and 13).

**Row order:** the rows keep the source file's order. That order follows the scraper's price searches, so row position
must never be used as a feature or to split the data. The later split shuffles the rows (spec section 13).

**Evidence:** spec sections 3, 4 and 12 (step 8).

In [27]:
FORBIDDEN_COLUMNS = [
    "price_per_sqft", "price_label", "description", "raw_json",
    "loc_avg_price_sqft", "loc_avg_price_lmt", "loc_price_quarter_prev", "loc_price_quarter_current",
    "loc_growth_quarterly_pct", "loc_locality_rank",
    "scraped_at",
]
DEFINITELY_REMOVE_COLUMNS = [
    "city", "is_Luxury", "detail_scraped", "has_rera", "legal_status", "infra_status", "loc_city_name", "loc_city_id",
    "bhk", "property_title", "coordinates", "loc_lat", "loc_long", "luxAmenMap", "lux_amen_desc", "loc_locality_name",
    "area_type", "loc_scraped_at",
]
NOT_NEEDED_METADATA_COLUMNS = ["detail_url"]

columns_to_drop = FORBIDDEN_COLUMNS + DEFINITELY_REMOVE_COLUMNS + NOT_NEEDED_METADATA_COLUMNS
missing_from_data = [column for column in columns_to_drop if column not in cleaned.columns]
assert missing_from_data == [], f"Columns not found: {missing_from_data}"

columns_before = cleaned.shape[1]
cleaned = cleaned.drop(columns=columns_to_drop)
print("Columns before:", columns_before, "| dropped:", len(columns_to_drop), "| after:", cleaned.shape[1])
print("  forbidden (leakage):", len(FORBIDDEN_COLUMNS))
print("  definitely remove:  ", len(DEFINITELY_REMOVE_COLUMNS))
print("  metadata not needed:", len(NOT_NEEDED_METADATA_COLUMNS))

Columns before: 102 | dropped: 30 | after: 72
  forbidden (leakage): 11
  definitely remove:   18
  metadata not needed: 1


In [28]:
# Metadata extracted from raw_json in step 1 (spec section 3h). Never features.
cleaned["project_id"] = raw.loc[cleaned.index, "psmid"]
cleaned["advertiser_id"] = raw.loc[cleaned.index, "oid"]
cleaned["land_area_unit"] = raw.loc[cleaned.index, "landAreaUnitD"]

# Price band for reporting only: exactly the bins used in notebook 13 (section 6)
PRICE_BOUNDS = [0, 50e5, 80e5, 120e5, 150e5, 175e5, 200e5, 300e5, 500e5, np.inf]
RANGE_NAMES = ["0-50 L", "50-80 L", "80 L-1.2 Cr", "1.2-1.5 Cr", "1.5-1.75 Cr",
               "1.75-2 Cr", "2-3 Cr", "3-5 Cr", "5 Cr+"]
cleaned["price_band"] = pd.cut(cleaned["price"], bins=PRICE_BOUNDS, labels=RANGE_NAMES,
                               include_lowest=True, right=True).astype(str)

print("Listings per price band (for reporting only):")
print(cleaned["price_band"].value_counts().reindex(RANGE_NAMES).to_string())
assert cleaned["price_band"].isin(RANGE_NAMES).all()

Listings per price band (for reporting only):
price_band
0-50 L         1571
50-80 L        2893
80 L-1.2 Cr    2931
1.2-1.5 Cr     2872
1.5-1.75 Cr    2145
1.75-2 Cr      2052
2-3 Cr         2696
3-5 Cr         1948
5 Cr+          1192


### Column roles

Every column in the cleaned file gets exactly one role, taken from spec section 3. The role file is the guard against
accidentally using a column as a feature. Only the columns marked **possible feature** may be considered in feature
engineering, and even those are not guaranteed to be used.

In [29]:
COLUMN_ROLES = {
    "target": ["price"],
    "possible feature (spec 3e)": [
        "property_type", "bedrooms", "bathrooms", "balconies", "floor", "total_floors", "furnishing", "possession",
        "age_of_property", "facing", "flooring_type", "additional_rooms", "amenities", "luxury_amenities",
        "prime_location", "posted_by", "transaction_type", "covered_area_sqft_converted", "locality_id",
        "locality_matched", "latitude", "longitude", "loc_locality_lat", "loc_locality_long", "loc_zip_code",
        "loc_review_count", "land_area_unit",
    ] + RATING_COLUMNS,
    "input only, not a feature (typed value in the advertiser's unit; use covered_area_sqft_converted)": [
        "covered_area_sqft", "covered_area_unit",
    ],
    "probably remove: kept, not planned as a feature (spec 3d)": [
        "project_name", "builder_name", "sub_locality", "locality_coords", "loc_rating_users_count", "nonluxAmenMap",
        "carpet_area_sqft", "power_status", "water_status", "parking", "is_rera",
    ],
    "metadata only: never a feature (spec 3f)": [
        "property_id", "image_url", "owner_name", "pmt_source", "posted_label", "listing_date", "locality",
        "loc_pros_1", "loc_pros_2", "loc_cons_1", "loc_introduction", "project_id", "advertiser_id",
    ],
    "created by cleaning: never a feature (spec 3g)": [
        "property_group_id", "locality_match_status", "recovered_locality_id_candidate",
        "r2_area_unreliable", "r3_floor_contradiction", "price_band",
    ],
}

role_rows = []
for role, columns in COLUMN_ROLES.items():
    for column in columns:
        role_rows.append({"column": column, "role": role})
column_roles = pd.DataFrame(role_rows)

# Every column must have exactly one role, and every role entry must be a real column
unassigned = [column for column in cleaned.columns if column not in list(column_roles["column"])]
not_in_data = [column for column in column_roles["column"] if column not in cleaned.columns]
print("Columns without a role:", unassigned)
print("Roles for columns that do not exist:", not_in_data)
print("Columns with two roles:", column_roles["column"][column_roles["column"].duplicated()].tolist())
assert unassigned == [] and not_in_data == [] and column_roles["column"].is_unique

# The forbidden columns must be gone
assert [column for column in FORBIDDEN_COLUMNS if column in cleaned.columns] == []

print()
print(column_roles["role"].value_counts().to_string())

Columns without a role: []
Roles for columns that do not exist: []
Columns with two roles: []

role
possible feature (spec 3e)                                                                           43
metadata only: never a feature (spec 3f)                                                             13
probably remove: kept, not planned as a feature (spec 3d)                                            11
created by cleaning: never a feature (spec 3g)                                                        6
input only, not a feature (typed value in the advertiser's unit; use covered_area_sqft_converted)     2
target                                                                                                1


## 9. Final validation, save, reload

**What:** run the final checks, save the cleaned dataset and the column roles, load the saved file again and compare it
with what was saved, and confirm that both source files are still byte-for-byte unchanged.

**Why:** a check that is only printed can be missed; each check below is also an `assert`, so the notebook stops if
anything is wrong. The results go into the audit file.

In [30]:
checks = {}

# Target
checks["no missing price"] = cleaned["price"].notna().all()

# No impossible-price row remains: none of the R1 property_ids is in the cleaned data,
# and the R1 rule re-evaluated on the cleaned rows (with the original values) finds nothing
r1_ids = set(listings.loc[r1_impossible_price, "property_id"])
checks["no R1 property_id remains"] = not cleaned["property_id"].isin(r1_ids).any()
checks["R1 rule finds 0 rows in the cleaned data"] = r1_impossible_price[cleaned.index].sum() == 0
checks["no price >= 100 Cr remains"] = (cleaned["price"] < 100e7).all()

# Rows
checks["row count = original - D1 - R1"] = len(cleaned) == original_row_count - 280 - 114
checks["property_id unique"] = cleaned["property_id"].is_unique

# Group ID
checks["property_group_id present"] = "property_group_id" in cleaned.columns
checks["property_group_id has no missing values"] = cleaned["property_group_id"].notna().all()
checks["property_group_id has no empty values"] = (cleaned["property_group_id"] != "").all()
checks["property_group_id unchanged since step 2"] = (
    cleaned["property_group_id"] == listings.loc[cleaned.index, "property_group_id"]).all()
single_ids = cleaned["property_group_id"][cleaned["property_group_id"].str.startswith("single ")]
checks["'single' group IDs unique"] = single_ids.is_unique

# Transaction types all kept
checks["all 5 transaction_type labels still present"] = (
    set(cleaned["transaction_type"].dropna()) == {"Resale", "New Property", "Other", "Rent"}
    and cleaned["transaction_type"].isna().any())

# Area, floor, missing representations
converted_area = cleaned["covered_area_sqft_converted"].dropna()
checks["no R2 row keeps an area"] = cleaned.loc[cleaned["r2_area_unreliable"], "covered_area_sqft_converted"].isna().all()
checks["no floor above total floors"] = (
    pd.to_numeric(cleaned["floor"].replace(FLOOR_WORDS), errors="coerce") > cleaned["total_floors"]).sum() == 0
checks["no latitude/longitude equal to 0"] = ((cleaned["latitude"] == 0) | (cleaned["longitude"] == 0)).sum() == 0
checks["no locality rating equal to 0"] = (cleaned[RATING_COLUMNS] == 0).sum().sum() == 0

# Leakage columns gone
checks["no forbidden column present"] = not any(column in cleaned.columns for column in FORBIDDEN_COLUMNS)

check_table = pd.Series(checks).to_frame("passed")
print(check_table.to_string())
assert check_table["passed"].all(), "A validation check failed"
print()
print("All validation checks passed.")

                                             passed
no missing price                               True
no R1 property_id remains                      True
R1 rule finds 0 rows in the cleaned data       True
no price >= 100 Cr remains                     True
row count = original - D1 - R1                 True
property_id unique                             True
property_group_id present                      True
property_group_id has no missing values        True
property_group_id has no empty values          True
property_group_id unchanged since step 2       True
'single' group IDs unique                      True
all 5 transaction_type labels still present    True
no R2 row keeps an area                        True
no floor above total floors                    True
no latitude/longitude equal to 0               True
no locality rating equal to 0                  True
no forbidden column present                    True

All validation checks passed.


In [31]:
cleaned.to_csv(CLEANED_CSV, index=False)
column_roles.to_csv(COLUMN_ROLES_CSV, index=False)
print("Saved:", CLEANED_CSV.relative_to(PROJECT_ROOT), cleaned.shape)
print("Saved:", COLUMN_ROLES_CSV.relative_to(PROJECT_ROOT))

# Load the saved file again and compare the important columns with what we saved
reloaded = pd.read_csv(CLEANED_CSV, low_memory=False)
reload_checks = {
    "reloaded: same number of rows": len(reloaded) == len(cleaned),
    "reloaded: same columns in the same order": list(reloaded.columns) == list(cleaned.columns),
    "reloaded: same property_id values": (reloaded["property_id"].values == cleaned["property_id"].values).all(),
    "reloaded: same property_group_id values": (
        reloaded["property_group_id"].values == cleaned["property_group_id"].values).all(),
    "reloaded: same price values": (reloaded["price"].values == cleaned["price"].values).all(),
    "reloaded: no missing price": reloaded["price"].notna().all(),
    "reloaded: property_group_id has no missing values": reloaded["property_group_id"].notna().all(),
}
print(pd.Series(reload_checks).to_frame("passed").to_string())
assert all(reload_checks.values()), "The saved file does not load back correctly"
checks.update(reload_checks)

Saved: data\processed\cleaned_listings.csv (20300, 76)
Saved: data\processed\cleaned_listings_column_roles.csv
                                                   passed
reloaded: same number of rows                        True
reloaded: same columns in the same order             True
reloaded: same property_id values                    True
reloaded: same property_group_id values              True
reloaded: same price values                          True
reloaded: no missing price                           True
reloaded: property_group_id has no missing values    True


In [32]:
md5_at_end = {"merged_listings.csv": file_md5(MERGED_CSV), "localities.csv": file_md5(LOCALITIES_CSV)}
print("MD5 at end:", md5_at_end)
source_unchanged = md5_at_end == md5_at_start == EXPECTED_MD5
checks["source files byte-for-byte unchanged"] = source_unchanged
assert source_unchanged, "A source file changed!"
print("Both source files are byte-for-byte unchanged.")

MD5 at end: {'merged_listings.csv': '7d5c4224a1864259922d683567550231', 'localities.csv': '8d9bb54dfc2fb49673e74b976ab73ace'}
Both source files are byte-for-byte unchanged.


## Cleaning audit summary

One table with the row counts of every step and the result of every validation check. It is saved to
`reports/cleaning_audit.csv`.

In [33]:
add_to_audit("final rows", len(cleaned), "data/processed/cleaned_listings.csv")
add_to_audit("final columns", cleaned.shape[1],
             f"{original_column_count} source - {len(columns_to_drop)} dropped + "
             f"{cleaned.shape[1] - (original_column_count - len(columns_to_drop))} added")
add_to_audit("unique property_group_id (cleaned rows)", cleaned["property_group_id"].nunique())

for check_name, passed in checks.items():
    add_to_audit(f"check: {check_name}", "passed" if passed else "FAILED")

audit_table = pd.DataFrame(audit)
AUDIT_CSV.parent.mkdir(exist_ok=True)
audit_table.to_csv(AUDIT_CSV, index=False)

pd.set_option("display.max_colwidth", 90)
print(audit_table.to_string(index=False))
print()
print("Saved:", AUDIT_CSV.relative_to(PROJECT_ROOT))

                                                            step  value                                                                            note
                                                   original rows  20694                                              data/processed/merged_listings.csv
                                                original columns     95                                                                                
                        rows removed because of transaction_type      0                             all rows are website sale listings (spec section 5)
                      unique property_group_id (all 20,694 rows)  19851                                              matches docs/duplicate_handling.md
                                rows removed: missing price (D1)    280                                                                       no target
                             rows removed: impossible price (R1)    114                 

## What was done, and what was deliberately not done

**Done (all from `docs/data_cleaning_decisions.md`):**
- `property_group_id` v2 built on the raw data and preserved for the later group-aware split.
- Rows removed only for missing price (D1) and impossible price (R1).
- All transaction types kept (`Rent` and `Other` included).
- Covered area converted to square feet with the true unit; unreliable areas (R2) set to missing, no unit guessed.
- Contradictory floors (R3) set to missing.
- Placeholder values ((0, 0) coordinates, rating 0, `{}`, `Na`) set to missing.
- Locality match status recorded; safe free-text matches recorded as candidates, **not** applied.
- Forbidden and definitely-removed columns dropped; every remaining column has a documented role.

**Not done here (later notebooks):**
- the group-aware train/test split on `property_group_id`
- imputation, unknown/rare-category encoding, rare-locality grouping, any locality price encoding (training data only)
- feature engineering (how area is used per property type, `"> 10"` bedrooms, parsing `possession`, amenities, …)
- the coordinate rule R4, which is only needed if geographic features are used
- the open questions in spec section 16 (e.g. applying the locality recovery, manual review of the ambiguous high-end listings)

**Reminders for the modelling stage:** `price_band`, `property_group_id`, `property_id`, `listing_date` and every other
column not marked "possible feature" in `cleaned_listings_column_roles.csv` must never be model inputs; the rows are in
scrape (≈ price) order, so always shuffle through the group-aware splitter; report errors per `price_band` as well as
overall.